# DNABERT (Original) with Real Genomic DNA

Tests DNABERT k-mer variants on **real genomic sequences** (human chr22).

## Purpose

- **Synthetic DNA (other notebook)**: Tests pure tokenization effects
- **Real DNA (this notebook)**: Ecological validity -- k-mers on real genome structure

---

In [ ]:
# Install Dependencies
print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops
import transformers, torch
print(f"transformers {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("Done!")

In [ ]:
# Configuration
import sys, os
sys.path.insert(0, '.')
PHASE = 'full'
OUTPUT_BASE = './results/dnabert_realdna_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {'n_sequences': 500, 'seq_length': 510, 'models': [
        'zhihan1996/DNA_bert_3', 'zhihan1996/DNA_bert_6'],
        'batch_size': 8, 'n_bootstrap': 0, 'snp_rates': [0.01, 0.05, 0.10]},
    'full': {'n_sequences': 10000, 'seq_length': 510, 'models': [
        'zhihan1996/DNA_bert_3', 'zhihan1996/DNA_bert_4',
        'zhihan1996/DNA_bert_5', 'zhihan1996/DNA_bert_6'],
        'batch_size': 8, 'n_bootstrap': 5, 'snp_rates': [0.01, 0.02, 0.05, 0.10]},
}
config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()} | Data: Real DNA (chr22) | Models: {len(config['models'])}")

In [ ]:
# Load Real Genomic DNA
import urllib.request, gzip, os
import numpy as np
CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')

def download_and_sample_genomic_dna(n_sequences, seq_length, seed=320):
    cache_file = f'{CACHE_DIR}/chr22_sample_{n_sequences}_{seq_length}.txt'
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sequences)} cached sequences")
        return sequences
    print("Downloading human chr22...")
    os.makedirs(CACHE_DIR, exist_ok=True)
    with urllib.request.urlopen(CHR22_URL) as response:
        with gzip.GzipFile(fileobj=response) as f:
            lines = f.read().decode('utf-8').split('\n')
            chr22_seq = ''.join(line.strip() for line in lines[1:] if line.strip())
    print(f"Downloaded chr22 ({len(chr22_seq):,} bp)")
    rng = np.random.default_rng(seed)
    sequences = []
    for _ in range(int(n_sequences * 1.5)):
        start = rng.integers(0, len(chr22_seq) - seq_length)
        seq = chr22_seq[start:start + seq_length].upper()
        if sum(1 for c in seq if c not in VALID_BASES) < seq_length * 0.1:
            seq_clean = ''.join(c if c in VALID_BASES else rng.choice(['A','C','G','T']) for c in seq)
            sequences.append(seq_clean)
        if len(sequences) >= n_sequences: break
    with open(cache_file, 'w') as f: f.write('\n'.join(sequences))
    return sequences[:n_sequences]

sequences = download_and_sample_genomic_dna(config['n_sequences'], config['seq_length'])
print(f"Sequences: {len(sequences)}, Length: {len(sequences[0])}")

In [ ]:
# DNA Perturbation Suite
from dataclasses import dataclass, field
DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}
@dataclass
class PerturbedSet:
    name: str; category: str; sequences: list
    params: dict = field(default_factory=dict); description: str = ''
def mutate_dna(sequence, mutation_rate, rng):
    seq = list(sequence)
    valid = [i for i, c in enumerate(seq) if c in DNA_BASES]
    n = max(1, int(len(valid) * mutation_rate))
    for pos in rng.choice(valid, size=min(n, len(valid)), replace=False):
        seq[pos] = rng.choice([b for b in DNA_BASES if b != seq[pos]])
    return ''.join(seq)
def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))
class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc
    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate*100)}pct"
            results[name] = PerturbedSet(name=name, category='snp',
                sequences=[mutate_dna(s, rate, self.rng) for s in sequences],
                params={'mutation_rate': rate})
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(name='reverse_complement',
                category='rc', sequences=[reverse_complement(s) for s in sequences])
        return results
suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print("Perturbation suite ready")

In [ ]:
# DNABERT Model Wrapper (same as Scaling notebook)
import torch, gc, re
from transformers import AutoTokenizer, AutoModel
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.synchronize()
def seq_to_kmer(sequence, k):
    return ' '.join(sequence[i:i+k] for i in range(len(sequence) - k + 1))
def get_kmer_size(model_name):
    match = re.search(r'bert_(\d+)', model_name)
    return int(match.group(1)) if match else 6
def make_dnabert_embedding_fn(model_name, batch_size=8):
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    k = get_kmer_size(model_name)
    print(f"  Device: {device}, K-mer: {k}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device).eval()
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Params: {n_params:.1f}M")
    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i + batch_size]
            if (i // batch_size + 1) % 20 == 0 or i // batch_size + 1 == n_batches:
                print(f"    Batch {i//batch_size+1}/{n_batches}", end='\r')
            kmer_seqs = [seq_to_kmer(s, k) for s in batch_seqs]
            tokens = tokenizer(kmer_seqs, return_tensors='pt', padding=True, truncation=True, max_length=512)
            tokens = {key: v.to(device) for key, v in tokens.items()}
            hidden = model(**tokens).last_hidden_state
            mask = tokens['attention_mask'].unsqueeze(-1)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            embeddings.append(pooled.cpu().numpy())
        print()
        return np.concatenate(embeddings, axis=0)
    return embed, model, tokenizer, n_params
print("DNABERT wrapper ready")

In [ ]:
# Evaluation Harness
from evaluation_harness import StabilityHarness
harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30, seed=320, max_samples=2500, n_bootstrap=config['n_bootstrap'])
print(f"Harness configured")

In [ ]:
# Run Experiment
import os, time, pandas as pd
from evaluation_harness import ModelReport
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print('=' * 70)
print(f'DNABERT + REAL GENOMIC DNA -- Phase: {PHASE.upper()}')
print('=' * 70)
reports = []; model_param_counts = []; all_detailed_rows = []
for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}\n[{model_idx+1}/{len(config['models'])}] {model_name}\n{'='*70}")
    start_time = time.time()
    embed_fn, model_obj, tokenizer_obj, n_params_m = make_dnabert_embedding_fn(model_name, config['batch_size'])
    model_param_counts.append(n_params_m)
    perturbed_sets = suite.run_all(sequences)
    safe_name = model_name.replace('/', '_').replace('-', '_') + '_realdna'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'
    if os.path.exists(cache_clean): embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings...')
        embeddings_clean = embed_fn(sequences); np.save(cache_clean, embeddings_clean)
    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert): perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])
    del model_obj, tokenizer_obj; cleanup_gpu()
    report = harness.evaluate_all_perturbations(model_name=model_name, embeddings_clean=embeddings_clean, perturbed_dict=perturbed_embeddings)
    reports.append(report)
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({'model': short_name, 'size_M': round(n_params_m, 1), 'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0), 'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0), 'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0)})
    print(f'Completed in {(time.time()-start_time)/60:.1f} min | Composite: {report.summary()["mean_composite_stability"]:.4f}')
df_detailed = pd.DataFrame(all_detailed_rows)
df_detailed.to_csv(f'{RESULTS_DIR}/dnabert_realdna_{PHASE}_detailed.csv', index=False)
print(f'\nSaved. EXPERIMENT COMPLETE.')

In [ ]:
# Visualization
import matplotlib.pyplot as plt
from evaluation_harness import compare_models
comparison = compare_models(reports, output_dir=RESULTS_DIR)
kmer_sizes = [get_kmer_size(r.model_name) for r in reports]
composites = [r.summary()['mean_composite_stability'] for r in reports]
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(kmer_sizes, composites, color=['tab:blue','tab:green','tab:orange','tab:red'][:len(kmer_sizes)], width=0.6)
ax.set_xlabel('K-mer Size'); ax.set_ylabel('Composite Stability')
ax.set_title('DNABERT on Real DNA: K-mer Tokenization Effect', fontweight='bold')
ax.set_xticks(kmer_sizes); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/dnabert_realdna_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()